# RAG DAY 2 — Real PDF + LLM Answers + Page Citations


In [2]:
# Goal: Load a real PDF, split into chunks, store in ChromaDB,
#       ask questions and get answers with page numbers

import os              # for environment variables (like API keys)
import re              # for text cleaning / regex (optional use)

from langchain_community.document_loaders import PyPDFLoader  
# loads and reads PDF files

from langchain_text_splitters import RecursiveCharacterTextSplitter  
# splits large text into smaller chunks

from sentence_transformers import SentenceTransformer  
# converts text into embeddings (vectors)

import chromadb  
# vector database to store and search embeddings

from groq import Groq  
# LLM API client (to generate answers)

c:\Users\rawat\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# STEP 1: Get a PDF to work with

In [3]:
print("=" * 55)
print("STEP 1: PDF Setup")
print("=" * 55)

PDF_PATH = "document.pdf"



STEP 1: PDF Setup


# STEP 2: Load the PDF


In [4]:
print("\n" + "=" * 55)
print("STEP 2: Loading PDF")
print("=" * 55)

# PyPDFLoader reads your PDF page by page
# Each page becomes a Document object with:
# - page_content: the text on that page
# - metadata: { source: filename, page: page_number }

loader=PyPDFLoader(PDF_PATH)
pages=loader.load()
print(f"✅ PDF loaded!")
print(f"Total pages: {len(pages)}")
print(f"\nPage 1 preview:")
print(pages[0].page_content[:300]+"...")
print(f"\nPage 1 metadata :{pages[0].metadata}")
# 🔑 metadata["page"] gives us the page number
# This is what we use for citations later!





STEP 2: Loading PDF
✅ PDF loaded!
Total pages: 16

Page 1 preview:
LA 5 
 
 
 
Name – Garv Rawat 
Registration Number - 23BLC1058 
Faculty Name - Dr. Brindha S 
Course - Software Engineering Lab 
(BCSE301P) 
School of Electronics Engineering (SENSE)...

Page 1 metadata :{'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-09T21:17:51+05:30', 'author': 'dsplab', 'moddate': '2026-04-09T21:17:51+05:30', 'source': 'document.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}


# STEP 3: Split into Chunks

In [5]:
print("\n" + "=" * 55)
print("STEP 3: Splitting into Chunks")
print("=" * 55)
# Why split? LLMs have a context limit — they can't read
# a 100-page PDF all at once. We split into small chunks
# so we only send the RELEVANT parts to the LLM.

# RecursiveCharacterTextSplitter tries to split on:
# 1. Paragraphs (\n\n) first
# 2. Then sentences (. )
# 3. Then words ( )
# 4. Then characters
# This keeps sentences intact as much as possible

splitter=RecursiveCharacterTextSplitter(
chunk_size=500,# max 500 characters per chunk
chunk_overlap=50,# overlap so context isn't lost at boundaries
length_function = len,
)#Chunks keep the page number from where they came store metadaat

chunks=splitter.split_documents(pages)
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks)} chars")

print(f"\nChunk 1:")
print(f"  Text: {chunks[0].page_content[:200]}...")
print(f"  Metadata: {chunks[0].metadata}")

print(f"\nChunk 2:")
print(f"  Text: {chunks[1].page_content[:200]}...")
print(f"  Metadata: {chunks[1].metadata}")
# 🔑 Each chunk keeps its metadata including page number
# So when we retrieve a chunk, we know which page it came from





STEP 3: Splitting into Chunks
Total chunks: 25
Average chunk size: 340.0 chars

Chunk 1:
  Text: LA 5 
 
 
 
Name – Garv Rawat 
Registration Number - 23BLC1058 
Faculty Name - Dr. Brindha S 
Course - Software Engineering Lab 
(BCSE301P) 
School of Electronics Engineering (SENSE)...
  Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-09T21:17:51+05:30', 'author': 'dsplab', 'moddate': '2026-04-09T21:17:51+05:30', 'source': 'document.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}

Chunk 2:
  Text: Package Diagram 
1. Overview of the Package Diagram 
The Stock Trading & Risk Management System Package Diagram represents the high-
level organization of the system by grouping related components int...
  Metadata: {'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-04-09T21:17:51+05:30', 'author': 'dsplab', 'moddate': '2026-04-09T21:17:51+05:30', 'source': 'document.pdf', 'total_pages': 

# STEP 4: Create Embeddings + Store in ChromaDB

In [6]:
print("\n" + "=" * 55)
print("STEP 4: Embeddings + ChromaDB Storage")
print("=" * 55)

print("Loading embedding model...")
embedder=SentenceTransformer("all-MiniLM-L6-v2")
print("Model ready")

#generate embeddings for all chunks
print(f"Generating embeddings for {len(chunks)} chunks...")
texts=[]
for chunk in chunks:
    texts.append(chunk.page_content)
embeddings =embedder.encode(texts,show_progress_bar=True)
print("Done!", embeddings.shape)
# 25 → number of chunks (rows)
# 384 → size of each embedding (columns)

print("\nStoring in ChromaDB...")
client=chromadb.PersistentClient(path="./rag_chrom_db")#This is the folder where your data is stored
#It creates a ChromaDB client that saves data to disk


# Delete existing collection if it exists (fresh start)
try:
    client.delete_collection("pdf_documents")
except:
    pass

collection=client.create_collection("pdf_documents")
collection.add(
    documents=texts,#documentt=[[doc1,doc2]]
    embeddings=embeddings.tolist(),
    metadatas=[
        {"page":chunk.metadata.get("page",0)+1,#0-indexed
        "source":chunk.metadata.get("source",PDF_PATH),
        "chunk":i,
        }
        for i, chunk in enumerate(chunks)#enumerate() lets you loop + get index at the same time
    ],
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)
print(f"✅ Stored {collection.count()} chunks in ChromaDB!")




STEP 4: Embeddings + ChromaDB Storage
Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5035.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready
Generating embeddings for 25 chunks...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.77it/s]


Done! (25, 384)

Storing in ChromaDB...
✅ Stored 25 chunks in ChromaDB!


# STEP 5: Retrieval — Find Relevant Chunks

In [7]:
print("\n" + "=" * 55)
print("STEP 5: Retrieval")
print("=" * 55)

def retrieve_chunks(question,n_results=3):
    """Find most relevant chunks for a question"""
    q_embedding=embedder.encode(question).tolist()
    results=collection.query(
        query_embeddings=q_embedding,
        n_results=n_results,
        include=["documents","metadatas","distances"]
    )
    chunk_found=[]
    for i in range(len(results["documents"][0])):
        chunk_found.append({
            "text":results["documents"][0][i],
            "page":results["metadatas"][0][i]["page"],
            "source":results["metadatas"][0][i]["source"],
            "relevance":round(1-results["distances"][0][i],2)
        })
    return chunk_found

question = "What is stock trading package diagram"
chunks_found = retrieve_chunks(question)
for i,chunk in enumerate(chunks_found):
     print(f"\n  Chunk {i+1} — Page {chunk['page']} (relevance: {chunk['relevance']})")
     print(f"  {chunk['text'][:150]}...")



STEP 5: Retrieval

  Chunk 1 — Page 2 (relevance: 0.69)
  Package Diagram 
1. Overview of the Package Diagram 
The Stock Trading & Risk Management System Package Diagram represents the high-
level organizatio...

  Chunk 2 — Page 4 (relevance: 0.68)
  5. Conclusion 
The package diagram provides a structured view of the stock trading system using layered 
architecture. It ensures scalability, securit...

  Chunk 3 — Page 5 (relevance: 0.47)
  Component Diagram 
1. Overview of the Component Diagram 
The Stock Trading System Component Diagram illustrates how different components 
interact to ...


# Summary


In [8]:
# ================================
# 🧠 RAG PIPELINE (FULL SUMMARY)
# ================================

# STEP 1: Load PDF
# - Use PyPDFLoader to read PDF
# - Output: list of pages (docs)
# - Each page has:
#     - page_content (text)
#     - metadata (page number, source)

# loader = PyPDFLoader("document.pdf")
# docs = loader.load()


# STEP 2: Split into chunks
# - Pages are too large → split into smaller chunks
# - Improves retrieval accuracy
# - Each chunk STILL keeps original page number

# splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
# chunks = splitter.split_documents(docs)


# STEP 3: Create embeddings
# - Convert text → numerical vectors
# - Same meaning → similar vectors

# texts = []
# for chunk in chunks:
#     texts.append(chunk.page_content)

# embeddings = embedder.encode(texts)


# STEP 4: Initialize ChromaDB
# - Persistent database → saves data to disk
# - Can reuse later without recomputing

# client = chromadb.PersistentClient(path="./rag_chroma_db")
# collection = client.get_or_create_collection(name="pdf_data")


# STEP 5: Prepare metadata + IDs
# - Metadata stores extra info:
#     - page number
#     - source file
#     - chunk index
# - IDs are unique names for each chunk

# metadatas = []
# ids = []

# for i, chunk in enumerate(chunks):
#     metadatas.append({
#         "page": chunk.metadata.get("page", 0) + 1,  # convert to 1-based index
#         "source": "document.pdf",
#         "chunk": i
#     })
#     ids.append(f"chunk_{i}")


# STEP 6: Store in ChromaDB
# - Store:
#     documents → text
#     embeddings → vectors
#     metadata → page info
#     ids → unique identifiers

# collection.add(
#     documents=texts,
#     embeddings=embeddings,
#     metadatas=metadatas,
#     ids=ids
# )


# ================================
# 🔍 QUERY (RETRIEVAL PART)
# ================================

# STEP 7: Ask a question
# - Convert question → embedding
# - Compare with stored embeddings
# - Retrieve most similar chunks

# results = collection.query(
#     query_texts=["What is machine learning?"],
#     n_results=3
# )


# STEP 8: View results
# - Get text + metadata
# - Show page numbers

# docs = results["documents"][0]
# metas = results["metadatas"][0]

# for doc, meta in zip(docs, metas):
#     print(f"Page {meta['page']}: {doc}")


# ================================
# 🧠 FULL FLOW (MENTAL MODEL)
# ================================

# PDF → Pages → Chunks → Embeddings → Store in DB
#                                      ↓
#                              User Question
#                                      ↓
#                         Convert to embedding
#                                      ↓
#                      Compare with stored vectors
#                                      ↓
#                         Return best matching chunks


# ================================
# ⚡ ONE-LINE SUMMARY
# ================================

# RAG = Convert PDF → chunks → embeddings → store → retrieve relevant chunks for a question

# STEP 6: Generation — Get LLM Answer with Citations

In [11]:
print("\n" + "=" * 55)
print("STEP 6: LLM Answer with Citations")
print("=" * 55)

# Make sure GROQ_API_KEY is in your .env or set it here:
# os.environ["GROQ_API_KEY"] = "your-key-here"
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "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")
groq_client = Groq(api_key=GROQ_API_KEY)

def ask_question(question, n_chunks=3):
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks
    2. Build context with page numbers
    3. Send to LLM
    4. Return answer with citations
    """
    print(f"\nQuestion: '{question}'")
    print("=" * 55)

    # Step 1: Retrieve
    chunks_found = retrieve_chunks(question, n_chunks)

    # Step 2: Build context with page citations
    context = ""
    for chunk in chunks_found:
        context += f"[Page {chunk['page']}]: {chunk['text']}\n\n"

    # Step 3: Build prompt
    # 🔑 We explicitly tell the LLM to:
    # a) Only use the provided context
    # b) Include page numbers in the answer
    # c) Say "I don't know" if context doesn't help
    prompt = f"""You are a helpful assistant that answers questions based ONLY on the provided context.
Always cite the page number when you use information from the context.
If the context doesn't contain enough information, say "I don't have enough information in the document to answer this."

Context from document:
{context}

Question: {question}

Answer (include page numbers as citations like "According to Page X..."):"""

    # Step 4: Call Groq LLM
    try:
        response = groq_client.chat.completions.create(
            model    = "llama-3.3-70b-versatile",
            messages = [{"role": "user", "content": prompt}],
            max_tokens = 500,
            temperature = 0.1,  # low temperature = more factual, less creative
        )
        answer = response.choices[0].message.content

    except Exception as e:
        answer = f"LLM error: {e}\nCheck your GROQ_API_KEY"

    # Step 5: Show answer + sources used
    print(f"\nAnswer:\n{answer}")
    print(f"\nSources used:")
    for chunk in chunks_found:
        print(f"  → Page {chunk['page']} (relevance: {chunk['relevance']})")

    return {
        "question": question,
        "answer":   answer,
        "sources":  chunks_found
    }
result1 = ask_question("What is component diagram for stock trading system")

result4 = ask_question("What is the weather like today?")
# Should say it doesn't have enough information



STEP 6: LLM Answer with Citations

Question: 'What is component diagram for stock trading system'

Answer:
According to Page 5, the Component Diagram for the Stock Trading System illustrates how different components interact to perform trading operations, risk evaluation, and portfolio management, highlighting modular design and service interaction. It consists of major components such as the User Interface, which serves as the entry point for traders, and the REST API / API Gateway, which handles communication and routes requests [Page 5]. Additionally, the Core Backend Services include the Auth Service [Page 5].

Sources used:
  → Page 5 (relevance: 0.75)
  → Page 4 (relevance: 0.53)
  → Page 2 (relevance: 0.45)

Question: 'What is the weather like today?'

Answer:
I don't have enough information in the document to answer this. The provided context does not mention the weather at all (Page 1, Page 6, Page 8).

Sources used:
  → Page 8 (relevance: -0.81)
  → Page 6 (relevance: -0.93)

# ============================================================
# STEP 7: Full Pipeline Function
# Clean version you'll use in your Flask API
# ============================================================

In [14]:

print("\n" + "=" * 55)
print("STEP 7: Clean Pipeline Function")
print("=" * 55)

class RAGPipeline:
    """
    Clean RAG pipeline class.
    This is what your Flask API will use.
    """

    def __init__(self, pdf_path, collection_name="documents"):
        print(f"Initializing RAG pipeline for: {pdf_path}")

        # Load and chunk PDF
        loader   = PyPDFLoader(pdf_path)
        pages    = loader.load()
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=50
        )
        self.chunks = splitter.split_documents(pages)

        # Create embeddings
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        texts         = [c.page_content for c in self.chunks]
        embeddings    = self.embedder.encode(texts)

        # Store in ChromaDB
        self.client = chromadb.PersistentClient(path="./rag_db")
        try:
            self.client.delete_collection(collection_name)
        except:
            pass
        self.collection = self.client.create_collection(collection_name)
        self.collection.add(
            documents  = texts,
            embeddings = embeddings.tolist(),
            metadatas  = [
                {"page": c.metadata.get("page", 0) + 1}
                for c in self.chunks
            ],
            ids = [f"chunk_{i}" for i in range(len(self.chunks))]
        )

        # Groq client
        self.groq = Groq(api_key="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")
        print(f"✅ Ready! Indexed {len(self.chunks)} chunks from {len(pages)} pages")

    def ask(self, question, n_chunks=3):
        """Ask a question and get answer with citations"""

        # Retrieve
        q_emb   = self.embedder.encode([question]).tolist()
        results = self.collection.query(
            query_embeddings = q_emb,
            n_results        = n_chunks,
            include          = ["documents", "metadatas", "distances"]
        )

        # Build context
        context = ""
        sources = []
        for i in range(len(results["documents"][0])):
            page = results["metadatas"][0][i]["page"]
            text = results["documents"][0][i]
            relevance = round(1 - results["distances"][0][i], 2)
            context += f"[Page {page}]: {text}\n\n"
            sources.append({"page": page, "relevance": relevance})

        # Generate answer
        prompt = f"""Answer based ONLY on this context. Cite page numbers.
If context is insufficient, say so.

Context:
{context}

Question: {question}
Answer:"""

        response = self.groq.chat.completions.create(
            model      = "llama-3.3-70b-versatile",
            messages   = [{"role": "user", "content": prompt}],
            max_tokens = 400,
            temperature= 0.1
        )

        return {
            "answer":  response.choices[0].message.content,
            "sources": sources,
            "question": question
        }
    

rag = RAGPipeline(PDF_PATH)
result = rag.ask("What types of machine learning exist?")
print(f"\nQ: {result['question']}")
print(f"A: {result['answer']}")
print(f"Sources: {result['sources']}")



STEP 7: Clean Pipeline Function
Initializing RAG pipeline for: document.pdf


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10038.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Ready! Indexed 25 chunks from 16 pages

Q: What types of machine learning exist?
A: The context is insufficient to answer this question. There is no mention of machine learning on any of the provided pages (Page 5 and Page 8).
Sources: [{'page': 8, 'relevance': -0.53}, {'page': 5, 'relevance': -0.6}, {'page': 8, 'relevance': -0.63}]


# DAY 2 CHALLENGE

In [ ]:
print("\n" + "=" * 55)
print("DAY 2 CHALLENGE")
print("=" * 55)

print("""
1. Use your OWN PDF — any PDF you have (notes, textbook,
   article). Change PDF_PATH and run the whole pipeline.
   Does it find the right answers?

   # Answer:
   # Yes, if the PDF contains the information,
   # the RAG pipeline usually retrieves correct answers.
   # If answers are wrong, retrieval may have failed
   # or chunk size may be poor.

2. Change chunk_size from 500 to 200, retrain.
   Then ask the same question. Is the answer better or worse?
   What does this tell you about chunk size choice?

   # Answer:
   # Smaller chunks (200) give more precise retrieval
   # because less unrelated text is included.
   # But too small chunks may lose context.
   # Larger chunks give more context but may include noise.
   # So chunk size is a tradeoff between:
   # precision vs context.

3. Change n_chunks from 3 to 1 in ask_question().
   Does the answer quality drop? Why?

   # Answer:
   # Usually yes.
   # With only 1 chunk, the LLM gets less information.
   # Important context may be missing.
   # Using multiple chunks improves answer quality
   # because the model sees more relevant content.

4. Ask a question that is NOT in your document.
   Does the LLM correctly say it doesn't know?
   Or does it hallucinate an answer?
   This is called "hallucination testing" — important in ML jobs.

   # Answer:
   # Sometimes the LLM still invents answers.
   # This is called hallucination.
   # Good prompts reduce hallucinations:
   # "Answer only from context."
   # But hallucinations can still happen.
   # Testing this is very important in production AI systems.

5. Most important — explain in your own words:
   Why do we need BOTH the embedding model AND the LLM?
   What does each one do that the other can't?

   # Answer:
   # Embedding model:
   # Converts text into vectors/numbers.
   # Used for semantic similarity search.
   # It finds relevant chunks from the database.

   # LLM:
   # Reads retrieved chunks and generates
   # human-like natural language answers.

   # Embedding model cannot generate answers.
   # LLM cannot efficiently search huge databases.
   # Together they form a complete RAG system.
""")



DAY 2 CHALLENGE

1. Use your OWN PDF — any PDF you have (notes, textbook,
   article). Change PDF_PATH and run the whole pipeline.
   Does it find the right answers?

2. Change chunk_size from 500 to 200, retrain.
   Then ask the same question. Is the answer better or worse?
   What does this tell you about chunk size choice?

3. Change n_chunks from 3 to 1 in ask_question().
   Does the answer quality drop? Why?

4. Ask a question that is NOT in your document.
   Does the LLM correctly say it doesn't know?
   Or does it hallucinate an answer?
   This is called "hallucination testing" — important in ML jobs.

5. Most important — explain in your own words:
   Why do we need BOTH the embedding model AND the LLM?
   What does each one do that the other can't?

